# Lesson 14 — Branching Rules

## Learning objectives

1. Understand pseudocost branching.
2. Implement strong branching {cite:p}`Achterberg2005` inside a hand-rolled B&B harness.
3. Recognize reliability branching as a hybrid.
4. Recognize the cost-per-node vs branching-quality trade-off and articulate it from data.

## 1. The branching problem

After solving the LP relaxation, several variables may be fractional. *Which* one to branch on is the **branching rule**. A bad rule can blow up the tree by orders of magnitude {cite:p}`Achterberg2005`.

## 2. Most fractional / Most infeasible

Pick the $x_j$ closest to half-integer. Cheap, but a poor predictor of *useful* branches.

## 3. Pseudocost branching

For each variable $x_j$, track historical *per-unit* changes in LP value when branched up ($\Phi_j^+$) and down ($\Phi_j^-$). Up-branching pushes $x_j$ from its LP value $x_j^{LP}$ to $\lceil x_j^{LP} \rceil$, a step of $1 - f_j$ where $f_j = x_j^{LP} - \lfloor x_j^{LP} \rfloor$; down-branching takes a step of $f_j$. The expected LP-value gains are therefore $\Phi_j^+(1 - f_j)$ on the up child and $\Phi_j^-\, f_j$ on the down child. Score:

$$\text{score}(j) = \max(\Phi_j^+ (1 - f_j), \epsilon) \cdot \max(\Phi_j^- f_j, \epsilon)$$

Pick the variable with the largest score (product, so a *bad* child is heavily penalised). Cheap; uses information from the whole tree {cite:p}`Achterberg2005`.

## 4. Strong branching

For each candidate $x_j$, *actually solve* both child LPs (limited iterations) and record the LP value gains. Pick the $j$ with the best (e.g., product of) gains.

Expensive — solving 2 LPs per candidate per node. But on hard MILPs, the branching quality more than pays for itself {cite:p}`Achterberg2005`.

## 5. Reliability branching

Hybrid: use pseudocosts, but switch to strong branching for variables whose pseudocost is "unreliable" (fewer than $\eta_{\text{rel}}$ observations in each direction). Strong-branching probes are themselves recorded as pseudocost observations, so a variable "graduates" to pseudocost scoring after a few probes. The default in modern solvers {cite:p}`Achterberg2005,Achterberg2007`.

`discopt`'s monolithic Rust MILP engine uses it too: the Rust `TreeManager` scores candidates with the pseudocost *product* rule above, probes unreliable candidates with limited strong branching (in the current source, `crates/discopt-core/src/bnb/branching.rs` and `milp_driver.rs`: reliability threshold 8, at most 8 probed candidates per node, and probing only while the tree is still small), and falls back to most-fractional for variables with no history at all.

That engine is the one `nlp_solver="simplex"` selects (the default route hands a pure MILP to HiGHS, which has its own reliability branching). `discopt`'s MILP tree, which `milp_backend="native"` selects, lets you choose the rule with `branching_rule=`: `"pseudocost"` (its default: pseudocosts, most-fractional until a variable has observations), `"most_fractional"`, `"least_fractional"` or `"strong"`. Its `"strong"` is *full* strong branching: both child LPs of every fractional candidate, solved to optimality, scored by the product of the two degradations. Those probe LPs are counted in `r.solver_stats["branching/strong_probe_lps"]`, not in `r.node_count`. A rule changes only *which* variable is branched on, never a bound, so every rule must reach the same optimum.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

# Two 30-binary instances.
def knapsack():
    """One row, values = weights (subset-sum-like: the LP bound is weak)."""
    rng = np.random.default_rng(1)
    w = rng.integers(10, 100, size=30)
    m = do.Model("knapsack")
    x = m.binary("x", shape=(30,))
    m.subject_to(w @ x <= int(w.sum() / 2))
    m.maximize(w @ x)
    return m

def mdkp():
    """Five knapsack rows: several variables can be fractional at once."""
    rng = np.random.default_rng(0)
    A = rng.integers(5, 60, size=(5, 30))
    v = rng.integers(5, 60, size=30)
    m = do.Model("mdkp")
    x = m.binary("x", shape=(30,))
    m.subject_to(A @ x <= A.sum(axis=1) // 2)
    m.maximize(v @ x)
    return m

# By default a pure MILP goes to HiGHS, with discopt checking the certificate
# HiGHS returns. `milp_backend="native"` runs discopt's own MILP tree instead,
# and only there can you choose the rule. `milp_cuts=False` and `batch_size=1`
# keep the comparison about branching alone: no root cuts, one node at a time.
rules = ["most_fractional", "least_fractional", "pseudocost", "strong"]
for name, build in [("knapsack", knapsack), ("mdkp", mdkp)]:
    print(f"{name}: HiGHS route nodes = {build().solve().node_count}")
    for rule in rules:
        r = build().solve(milp_backend="native", branching_rule=rule,
                          milp_cuts=False, batch_size=1)
        probes = int(r.solver_stats.get("branching/strong_probe_lps", 0))
        print(f"  {rule:17s} nodes={r.node_count:4d}  probe LPs={probes:4d}  "
              f"z*={r.objective:.0f}  time={r.wall_time:.2f}s")


## 6. Branching directions

For each branching, we choose **down first** (explore $x_j \le \lfloor x_j^{LP} \rfloor$ first) or **up first**. Default is to follow the LP relaxation's *rounding*: if $x_j^{LP} = 0.7$, try up first.

## 7. Plunging

A heuristic: after expanding a node, dive depth-first into one child until you hit infeasibility or integrality. Tends to find incumbents fast. Most modern solvers do this opportunistically.

## 8. Comparing rules yourself

Read the output of the cell above. On the single-row knapsack every rule builds exactly the same tree: a one-row LP relaxation has at most one fractional variable, so there is nothing to choose. On the five-row instance the rules differ. Least-fractional roughly doubles the tree, and strong branching saves the most nodes but pays for them in probe LPs, so it is not the fastest. That is the cost-per-node vs branching-quality trade-off of §4.

`branching_rule=` picks among four fixed rules. It does not let you cap the strong-branching LPs, change the reliability threshold, or choose the branching direction, and there is no branching callback (`node_callback` only *observes* the search). So the exercises also have you write a small B&B harness (LP relaxations via `scipy.optimize.linprog`) where every choice is yours, and compare it with `discopt`'s own tree on the same instance. Use `m.solve()` as the reference optimum to check that your harness is correct.

## References
{cite:p}`Achterberg2005,Achterberg2007` are the canonical references for modern branching.